# Critical Radius and Like-Charge Molecular Oscillation in Weber Electrodynamics

This notebook demonstrates the sub-critical regime of Weber's force law, where two
**like-charged** (positive-positive) particles form a bound oscillatory "molecular"
state despite the Coulomb repulsion between them.

## Physics Summary

Weber's velocity-dependent force creates an effective inertial mass
$\mu_{\text{eff}}(r) = \mu(1 - \rho/r)$ that changes sign at the **critical radius**

$$\rho = \frac{q_1 q_2}{\mu c^2}$$

where $\mu = m_1 m_2 / (m_1 + m_2)$ is the reduced mass.

For two like charges ($q_1 q_2 > 0$), $\rho > 0$ and the dynamics bifurcate:

| Regime | $r_0$ vs $\rho$ | $\mu_{\text{eff}}$ | Dynamics |
|---|---|---|---|
| Distant | $r_0 > \rho$ | positive | Coulomb-like scattering |
| Critical | $r_0 = \rho$ | zero (singular) | Absolute barrier |
| Molecular | $r_0 < \rho$ | negative | Bound oscillation between $r_0$ and $r=0$ |

No continuous trajectory can cross $r = \rho$. Below $\rho$, the particles oscillate
radially with $\dot{r}^2 \to 2c^2$ as $r \to 0$. The energy $E = k/r_0 > 0$ is
**positive**, yet the system is permanently bound.

**Regularizability (Frauenfelder & Weber 2024):** Only head-on collisions
($\ell = 0$) are $C^0$-continuable at finite speed $\sqrt{2}\,c$. Spiralling
collisions ($\ell \neq 0$) reach $r = 0$ at infinite speed in finite time and
are **not** regularizable by any smooth coordinate-time transform. No periodic
orbits exist inside $\rho$.

We perform three simulations:
1. **Run 1**: Sub-critical radial oscillation ($r_0 = 0.05$, $\ell = 0$)
2. **Run 2**: Sub-critical radial oscillation at larger $r_0 = 0.10$ ($\ell = 0$)
3. **Run 3**: Super-critical scattering ($r_0 > \rho$, $p = 0$) for comparison

The collision at $r = 0$ is handled by a **collision bounce**: when the pair
separation drops below a threshold, the relative coordinate is reflected through
the origin ($\vec{q}_{\text{rel}} \to -\vec{q}_{\text{rel}}$), analytically
continuing the $C^0$ trajectory without integrating through the singularity.

**References**: Weber, Sixth Memoir (1871) Sections 9.8--9.17; Frauenfelder &
Weber, *Anal. Math. Phys.* **14**:31 (2024); see
`research/theory/CriticalRadiusAndLikeChargeAttraction.md` and
`research/theory/Regularization.md`.

In [ ]:
using WeberElectrodynamics
using LinearAlgebra
using Plots
using Printf

## 1. System Construction and Physical Parameters

In [ ]:
# Physical parameters (equal-mass, equal-charge, like-sign pair)
m1, m2 = 1.0, 1.0
q1, q2 = 1.0, 1.0   # both positive — like charges
c = 4.0
k = q1 * q2          # k > 0 (repulsive Coulomb)

# Derived quantities
M = m1 + m2
mu = m1 * m2 / M     # reduced mass

# Critical radius: rho = k / (mu * c^2)
rho = k / (mu * c^2)

# Minimum sub-critical energy: E_min = mu * c^2
E_min = mu * c^2

system = HamiltonianSystem(2, 2)

@printf("Particles:   %d\n", system.n_particles)
@printf("Dimensions:  %d\n", system.dims)
@printf("DOF:         %d\n", system.degrees_of_freedom)
@printf("\nPhysical parameters:\n")
@printf("  m1 = %.1f, m2 = %.1f  (M = %.1f, mu = %.4f)\n", m1, m2, M, mu)
@printf("  q1 = %.1f, q2 = %.1f  (k = q1*q2 = %.1f)\n", q1, q2, k)
@printf("  c  = %.1f\n", c)
@printf("\nCritical radius:  rho = k/(mu*c^2) = %.6f\n", rho)
@printf("Barrier energy:   E_min = mu*c^2 = %.4f\n", E_min)

## 2. Symbolic Hamiltonian

In [ ]:
system.hamiltonian_symbolic

## 3. Run 1: Sub-Critical Molecular Oscillation ($r_0 = 0.05$)

Place two like charges at the outer turning point of a sub-critical oscillation.
Positions in COM frame along the x-axis with **zero momenta** ($\ell = 0$).

- $r_0 = 0.05 < \rho = 0.125$ (well below the critical radius)
- $\vec{p}_1 = \vec{p}_2 = \vec{0}$ (at turning point)
- $E = k/r_0 = 20.0 > 0$ (positive energy, yet bound)
- **Collision bounce** at $r_{\text{bounce}} = 0.01$: reflects the relative
  coordinate through the origin, analytically continuing the $C^0$ collision.
  Energy is exactly preserved by the reflection.

In [ ]:
# Sub-critical separation
r0_1 = 0.05
@assert r0_1 < rho "r0 must be sub-critical (< rho = $rho)"

# COM frame positions along x-axis
q0_1 = [-(m2/M)*r0_1, 0.0,   # particle 1: x, y
          (m1/M)*r0_1, 0.0]   # particle 2: x, y

# Zero momenta at turning point (ℓ = 0)
p0_1 = [0.0, 0.0,
         0.0, 0.0]

# Energy at turning point: E = k/r0 (Weber potential = Coulomb when rdot=0)
E_1 = k / r0_1

# Period estimate: T ~ 2*sqrt(2)*r0/c (traversal time r0 -> 0 -> r0)
T_1 = 2 * sqrt(2) * r0_1 / c
n_periods = 2
tspan_1 = (0.0, n_periods * T_1)

# Fixed timestep: small enough for midpoint convergence near bounce radius
dt_1 = 1e-4

# Collision bounce radius: reflects relative coordinate through origin at r < bounce_r
bounce_r = 0.02

@printf("Run 1 -- Sub-critical radial (ℓ=0):\n")
@printf("  r0   = %.4f  (rho = %.4f, ratio r0/rho = %.3f)\n", r0_1, rho, r0_1/rho)
@printf("  E    = k/r0 = %.4f  (E_min = mu*c^2 = %.4f)\n", E_1, E_min)
@printf("  T    ~ %.6f  (2*sqrt(2)*r0/c)\n", T_1)
@printf("  dt   = %.6f  (fixed)\n", dt_1)
@printf("  bounce_r = %.4f\n", bounce_r)
@printf("  tspan = (0.0, %.4f)  (%d periods)\n", tspan_1[2], n_periods)

In [ ]:
prob_1 = HamiltonianProblem(system, tspan_1, q0_1, p0_1;
    masses=[m1, m2], charges=[q1, q2], c=c, dt=dt_1,
    regularization=RegularizationOptions(collision_bounce_radius=bounce_r))

In [ ]:
sol_1 = solve(prob_1)

@printf("Run 1 retcode: %s\n", sol_1.retcode)
@printf("  Timesteps: %d\n", length(sol_1.t))

In [ ]:
traj_1 = compute_trajectory_data(sol_1, 2, 2; stride=1)
energy_1 = compute_energy_timeseries(sol_1; stride=1)
forces_1 = compute_pair_force_timeseries(sol_1, (1, 2), 2, 2, [m1, m2], [q1, q2], c; stride=1)
momentum_1 = compute_momentum_timeseries(sol_1; stride=1)

@printf("Run 1 energy conservation:\n")
@printf("  E(0) = %.6f\n", energy_1.total_energy[1])
@printf("  Max global error (%%): %.4e\n", energy_1.statistics.global_error_percent_max)
@printf("  Max local error: %.4e\n", energy_1.statistics.local_error_max)
@printf("  Min separation: %.6e\n", minimum(forces_1.phase_space.separation_distance))
@printf("  Max |rdot|/c: %.6f  (theory limit: sqrt(2) = %.6f)\n",
    maximum(abs.(forces_1.phase_space.radial_velocity)) / c, sqrt(2))

### Run 1 Plots: Sub-Critical Molecular Oscillation

In [ ]:
plot_trajectories(traj_1)

In [ ]:
plot_energy(energy_1)

In [ ]:
plot_pair_energy(energy_1, (1, 2))

In [ ]:
plot_energy_errors(energy_1)

In [ ]:
plot_pair_forces(forces_1)

In [ ]:
plot_phase_space(forces_1)

In [ ]:
plot_momentum_errors(momentum_1)

## 4. Run 2: Sub-Critical Oscillation at Larger $r_0 = 0.10$

Same physics as Run 1 but starting closer to $\rho$. This shows how the
oscillation changes with $r_0/\rho$: at larger $r_0$, the collision speed at
$r = 0$ is higher and the oscillation period is longer.

**Why not $\ell \neq 0$?** Frauenfelder & Weber (2024, Theorem 2.1) proved
that spiralling collisions ($\ell \neq 0$) inside $\rho$ reach the origin at
**infinite speed** in finite time, making them non-regularizable by any smooth
coordinate-time transform. Only head-on collisions ($\ell = 0$) are
$C^0$-continuable at the finite collision speed $\sqrt{2}\,c$.

- $r_0 = 0.10 < \rho = 0.125$ (ratio $r_0/\rho = 0.8$, closer to the barrier)
- $E = k/r_0 = 10.0$ (lower energy than Run 1, closer to $E_{\min} = 8.0$)

In [ ]:
# Sub-critical separation closer to rho
r0_2 = 0.10
@assert r0_2 < rho "r0 must be sub-critical (< rho = $rho)"

# COM frame positions along x-axis (same geometry as Run 1)
q0_2 = [-(m2/M)*r0_2, 0.0,
          (m1/M)*r0_2, 0.0]

# Zero momenta at turning point (ℓ = 0)
p0_2 = [0.0, 0.0,
         0.0, 0.0]

# Energy: E = k/r0
E_2 = k / r0_2

# Period estimate
T_2 = 2 * sqrt(2) * r0_2 / c
tspan_2 = (0.0, n_periods * T_2)

# Same fixed timestep and bounce radius as Run 1
dt_2 = 1e-4

@printf("Run 2 -- Sub-critical radial (ℓ=0), larger r0:\n")
@printf("  r0     = %.4f  (rho = %.4f, ratio r0/rho = %.3f)\n", r0_2, rho, r0_2/rho)
@printf("  E      = k/r0 = %.4f  (E_min = mu*c^2 = %.4f)\n", E_2, E_min)
@printf("  T      ~ %.6f  (2*sqrt(2)*r0/c)\n", T_2)
@printf("  dt     = %.6f  (fixed)\n", dt_2)
@printf("  bounce_r = %.4f  (same as Run 1)\n", bounce_r)
@printf("  tspan  = (0.0, %.4f)  (%d periods)\n", tspan_2[2], n_periods)

In [ ]:
prob_2 = HamiltonianProblem(system, tspan_2, q0_2, p0_2;
    masses=[m1, m2], charges=[q1, q2], c=c, dt=dt_2,
    regularization=RegularizationOptions(collision_bounce_radius=bounce_r))

sol_2 = solve(prob_2)

@printf("Run 2 retcode: %s\n", sol_2.retcode)
@printf("  Timesteps: %d\n", length(sol_2.t))

In [ ]:
traj_2 = compute_trajectory_data(sol_2, 2, 2; stride=1)
energy_2 = compute_energy_timeseries(sol_2; stride=1)
forces_2 = compute_pair_force_timeseries(sol_2, (1, 2), 2, 2, [m1, m2], [q1, q2], c; stride=1)
momentum_2 = compute_momentum_timeseries(sol_2; stride=1)

@printf("Run 2 energy conservation:\n")
@printf("  E(0) = %.6f\n", energy_2.total_energy[1])
@printf("  Max global error (%%): %.4e\n", energy_2.statistics.global_error_percent_max)
@printf("  Min separation: %.6e\n", minimum(forces_2.phase_space.separation_distance))
@printf("  Max |rdot|/c: %.6f  (theory limit: sqrt(2) = %.6f)\n",
    maximum(abs.(forces_2.phase_space.radial_velocity)) / c, sqrt(2))

### Run 2 Plots: Sub-Critical Oscillation ($r_0 = 0.10$)

In [ ]:
plot_trajectories(traj_2)

In [ ]:
plot_energy(energy_2)

In [ ]:
plot_pair_energy(energy_2, (1, 2))

In [ ]:
plot_energy_errors(energy_2)

In [ ]:
plot_pair_forces(forces_2)

In [ ]:
plot_phase_space(forces_2)

In [ ]:
plot_momentum_errors(momentum_2)

## 5. Run 3: Super-Critical Scattering (Comparison)

Place two like charges at the turning point of a super-critical trajectory.
With $r_0 = 0.3 > \rho = 0.125$, the pair is in the **distant state** and scatters
after reaching closest approach.

- $r_0 = 0.3$ (above critical radius, $r_0/\rho = 2.4$)
- $E = k/r_0 \approx 3.33 < E_{\min} = 8.0$ (distant state confirmed)
- Zero momenta at turning point: the pair is momentarily at rest and the
  positive effective mass ($\mu_{\text{eff}} > 0$ for $r > \rho$) means
  Coulomb repulsion drives the particles apart.

In [ ]:
# Super-critical separation
r0_3 = 0.3
@assert r0_3 > rho "r0 must be super-critical (> rho = $rho)"

# COM frame positions
q0_3 = [-(m2/M)*r0_3, 0.0,
          (m1/M)*r0_3, 0.0]

# Zero momenta at turning point
p0_3 = [0.0, 0.0,
         0.0, 0.0]

# Energy
E_3 = k / r0_3

# Simulation parameters — long enough to see clear separation
tspan_3 = (0.0, 20.0)
dt_3 = 0.002

@printf("Run 3 -- Super-critical scattering:\n")
@printf("  r0    = %.4f  (rho = %.4f, ratio r0/rho = %.3f)\n", r0_3, rho, r0_3/rho)
@printf("  E     = k/r0 = %.6f  (E_min = %.4f)\n", E_3, E_min)
@printf("  State = distant (r0 > rho, E < E_min)\n")
@printf("  tspan = (0.0, %.1f)\n", tspan_3[2])
@printf("  dt    = %.4f\n", dt_3)

In [ ]:
prob_3 = HamiltonianProblem(system, tspan_3, q0_3, p0_3;
    masses=[m1, m2], charges=[q1, q2], c=c, dt=dt_3,
    regularization=RegularizationOptions(
        enabled=true, backend=:lifted_pair,
        r_on=0.05, r_off=0.10, max_substeps=512))

sol_3 = solve(prob_3)

@printf("Run 3 retcode: %s\n", sol_3.retcode)
@printf("  Timesteps: %d\n", length(sol_3.t))
@printf("  Pair steps: %d (lifted: %d)\n",
    sol_3.regularization.pair_steps,
    sol_3.regularization.lifted_pair_steps)
@printf("  Min encounter distance: %.6f\n", sol_3.regularization.min_encounter_distance)

In [ ]:
traj_3 = compute_trajectory_data(sol_3, 2, 2; stride=1)
energy_3 = compute_energy_timeseries(sol_3; stride=1)
forces_3 = compute_pair_force_timeseries(sol_3, (1, 2), 2, 2, [m1, m2], [q1, q2], c; stride=1)
momentum_3 = compute_momentum_timeseries(sol_3; stride=1)

@printf("Run 3 energy conservation:\n")
@printf("  E(0)  = %.6f\n", energy_3.total_energy[1])
@printf("  E(end)= %.6f\n", energy_3.total_energy[end])
@printf("  Max global error (%%): %.4e\n", energy_3.statistics.global_error_percent_max)
@printf("  Final separation: %.4f\n", forces_3.phase_space.separation_distance[end])
@printf("  Max |rdot|: %.6f  (bounded < sqrt(2*rho/r0)*c = %.6f)\n",
    maximum(abs.(forces_3.phase_space.radial_velocity)),
    sqrt(2 * rho / r0_3) * c)

### Run 3 Plots: Super-Critical Scattering

In [ ]:
plot_trajectories(traj_3)

In [ ]:
plot_energy(energy_3)

In [ ]:
plot_energy_errors(energy_3)

In [ ]:
plot_phase_space(forces_3)

In [ ]:
plot_momentum_errors(momentum_3)

## 6. Comparative Analysis

### Phase Space Overlay

Compare the phase space portraits of all three runs to visualise the fundamental
difference between the molecular (bound) and distant (scattering) regimes.

In [ ]:
plt_phase = plot(;
    title = "Phase Space (r, rdot): Molecular vs Scattering",
    xlabel = "Separation r",
    ylabel = "Radial velocity rdot",
    legend = :outertopright,
    framestyle = :box,
    grid = true, gridalpha = 0.2,
    size = (1200, 1000),
)

# Run 1: sub-critical radial (closed loop, r0=0.05)
plot!(plt_phase,
    forces_1.phase_space.separation_distance,
    forces_1.phase_space.radial_velocity,
    label = "Run 1: molecular (r0=0.05)",
    linewidth = 2, color = :steelblue)

# Run 2: sub-critical radial (closed loop, r0=0.10)
plot!(plt_phase,
    forces_2.phase_space.separation_distance,
    forces_2.phase_space.radial_velocity,
    label = "Run 2: molecular (r0=0.10)",
    linewidth = 2, color = :forestgreen)

# Run 3: super-critical scattering (open curve)
plot!(plt_phase,
    forces_3.phase_space.separation_distance,
    forces_3.phase_space.radial_velocity,
    label = "Run 3: scattering (r0=0.30)",
    linewidth = 2, color = :firebrick)

# Mark critical radius
vline!(plt_phase, [rho],
    linestyle = :dash, linewidth = 2, color = :black,
    label = @sprintf("rho = %.3f (critical radius)", rho))

# Mark sqrt(2)*c velocity limit
hline!(plt_phase, [sqrt(2)*c, -sqrt(2)*c],
    linestyle = :dot, linewidth = 1, color = :gray,
    label = @sprintf("+/- sqrt(2)*c = +/- %.2f", sqrt(2)*c))

plt_phase

### Separation Distance vs Time

In [ ]:
plt_sep = plot(;
    title = "Pair Separation r(t): Oscillation vs Scattering",
    xlabel = "Time t",
    ylabel = "Separation r",
    legend = :outertopright,
    framestyle = :box,
    grid = true, gridalpha = 0.2,
    size = (1200, 600),
)

plot!(plt_sep, forces_1.phase_space.t, forces_1.phase_space.separation_distance,
    label = "Run 1: molecular (r0=0.05)", linewidth = 1.5, color = :steelblue)
plot!(plt_sep, forces_2.phase_space.t, forces_2.phase_space.separation_distance,
    label = "Run 2: molecular (r0=0.10)", linewidth = 1.5, color = :forestgreen)
plot!(plt_sep, forces_3.phase_space.t, forces_3.phase_space.separation_distance,
    label = "Run 3: scattering (r0=0.30)", linewidth = 1.5, color = :firebrick)

hline!(plt_sep, [rho],
    linestyle = :dash, linewidth = 2, color = :black,
    label = @sprintf("rho = %.3f", rho))

plt_sep

### Energy Comparison

In [ ]:
plt_energy = plot(;
    title = "Energy Timeseries: Molecular vs Scattering",
    xlabel = "Time t",
    ylabel = "Energy",
    legend = :outertopright,
    framestyle = :box,
    grid = true, gridalpha = 0.2,
    size = (1200, 800),
)

# Run 1 energies
plot!(plt_energy, energy_1.t, energy_1.total_energy,
    label = "Run 1: H (total)", linewidth = 2, color = :steelblue)
plot!(plt_energy, energy_1.t, energy_1.kinetic_energy,
    label = "Run 1: KE", linewidth = 1, color = :steelblue, linestyle = :dash, alpha = 0.6)

# Run 3 energies
plot!(plt_energy, energy_3.t, energy_3.total_energy,
    label = "Run 3: H (total)", linewidth = 2, color = :firebrick)
plot!(plt_energy, energy_3.t, energy_3.kinetic_energy,
    label = "Run 3: KE", linewidth = 1, color = :firebrick, linestyle = :dash, alpha = 0.6)

# Mark E_min = mu*c^2
hline!(plt_energy, [E_min],
    linestyle = :dot, linewidth = 1.5, color = :black,
    label = @sprintf("E_min = mu*c^2 = %.1f", E_min))

plt_energy

## 7. Summary and Diagnostics

In [ ]:
println("=" ^ 80)
println("CRITICAL RADIUS DEMONSTRATION -- SUMMARY")
println("=" ^ 80)

@printf("\nPhysical parameters:\n")
@printf("  m1 = m2 = %.1f, q1 = q2 = %.1f, c = %.1f\n", m1, q1, c)
@printf("  mu = %.4f, k = %.1f\n", mu, k)
@printf("  Critical radius rho = k/(mu*c^2) = %.6f\n", rho)
@printf("  Barrier energy E_min = mu*c^2 = %.4f\n", E_min)
@printf("  Collision bounce radius = %.4f\n", bounce_r)

println("\n" * "-" ^ 80)
@printf("%-20s  %-15s  %-15s  %-15s\n", "", "Run 1 (r0=0.05)", "Run 2 (r0=0.10)", "Run 3 (scatter)")
println("-" ^ 80)

@printf("%-20s  %-15.4f  %-15.4f  %-15.4f\n", "r0", r0_1, r0_2, r0_3)
@printf("%-20s  %-15.3f  %-15.3f  %-15.3f\n", "r0/rho", r0_1/rho, r0_2/rho, r0_3/rho)
@printf("%-20s  %-15s  %-15s  %-15s\n", "Regime", "molecular", "molecular", "distant")
@printf("%-20s  %-15.6f  %-15.6f  %-15.6f\n", "E = H(0)",
    energy_1.total_energy[1], energy_2.total_energy[1], energy_3.total_energy[1])

min_r1 = minimum(forces_1.phase_space.separation_distance)
min_r2 = minimum(forces_2.phase_space.separation_distance)
min_r3 = minimum(forces_3.phase_space.separation_distance)
@printf("%-20s  %-15.6e  %-15.6e  %-15.6f\n", "Min separation", min_r1, min_r2, min_r3)

max_rdot1 = maximum(abs.(forces_1.phase_space.radial_velocity))
max_rdot2 = maximum(abs.(forces_2.phase_space.radial_velocity))
max_rdot3 = maximum(abs.(forces_3.phase_space.radial_velocity))
@printf("%-20s  %-15.6f  %-15.6f  %-15.6f\n", "Max |rdot|/c", max_rdot1/c, max_rdot2/c, max_rdot3/c)

@printf("%-20s  %-15.4e  %-15.4e  %-15.4e\n", "Energy drift (%)",
    energy_1.statistics.global_error_percent_max,
    energy_2.statistics.global_error_percent_max,
    energy_3.statistics.global_error_percent_max)

@printf("%-20s  %-15s  %-15s  %-15s\n", "Retcode",
    string(sol_1.retcode), string(sol_2.retcode), string(sol_3.retcode))

println("-" ^ 80)

@printf("\nTheoretical checks:\n")
@printf("  Run 1 max |rdot|/c = %.4f  (theory: -> sqrt(2) = %.4f as r->0)\n", max_rdot1/c, sqrt(2))
@printf("  Run 2 max |rdot|/c = %.4f  (theory: -> sqrt(2) = %.4f as r->0)\n", max_rdot2/c, sqrt(2))
@printf("  Run 3 max |rdot|/c = %.4f  (theory: < sqrt(2*rho/r0) = %.4f)\n",
    max_rdot3/c, sqrt(2*rho/r0_3))
@printf("  Run 1 bound: max separation = %.6f <= r0 = %.6f  %s\n",
    maximum(forces_1.phase_space.separation_distance), r0_1,
    maximum(forces_1.phase_space.separation_distance) <= r0_1 + 1e-6 ? "PASS" : "FAIL")
@printf("  Run 2 bound: max separation = %.6f <= r0 = %.6f  %s\n",
    maximum(forces_2.phase_space.separation_distance), r0_2,
    maximum(forces_2.phase_space.separation_distance) <= r0_2 + 1e-6 ? "PASS" : "FAIL")

## 8. Conclusions

This notebook demonstrated the three qualitative regimes of the two-body like-charge
Weber system:

1. **Sub-critical molecular oscillation at $r_0 = 0.05$** (Run 1): With
   $r_0/\rho = 0.4$, the two positive charges form a bound oscillatory pair.
   Despite positive Coulomb repulsion and positive total energy ($E = 20$), the
   velocity-dependent Weber force creates an effective attraction below $\rho$.
   The particles oscillate radially between $r_0$ and $r \approx 0$ (truncated at
   the bounce radius). The maximum radial velocity approaches $\sqrt{2} \cdot c$
   as the bounce radius decreases. The particles never escape past $\rho$.

2. **Sub-critical molecular oscillation at $r_0 = 0.10$** (Run 2): Starting
   closer to $\rho$ ($r_0/\rho = 0.8$) gives a lower energy ($E = 10$, closer to
   $E_{\min} = \mu c^2 = 8$) and a longer oscillation period. Both runs confirm
   the permanent bound state.

3. **Super-critical scattering** (Run 3): With $r_0 = 0.3 > \rho$, the pair is in
   the distant (Coulomb-like) regime. After reaching the turning point, the particles
   separate to infinity. The maximum velocity is bounded below $\sqrt{2} \cdot c$.

The critical radius $\rho$ acts as an impenetrable barrier between these regimes.

**Collision handling**: The $r = 0$ singularity is handled by a collision bounce that
reflects the relative coordinate through the origin. This is physically exact for the
$\ell = 0$ case, where the collision is $C^0$-continuable at finite speed $\sqrt{2}\,c$
(Frauenfelder & Weber 2024). The $\ell \neq 0$ case is **not** regularizable: spiralling
collisions reach the origin at infinite speed, and no periodic orbits exist inside $\rho$.